# DQNet — Dynamic Quantization on ResNet-20 (CIFAR-10)

Trains a ResNet-20 with a lightweight bit-controller that assigns per-layer bit-widths (from {2, 4, 8}) on a per-image basis. The combined loss balances classification accuracy against BitOPs cost.


In [ ]:
!git clone https://github.com/MrRoboto11102003/Quantization_project.git 2>/dev/null; true
import sys
sys.path.append('Quantization_project')

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time, numpy as np, matplotlib.pyplot as plt
from DQ_resnet import dq_resnet20, compute_bitops, max_bitops, BIT_OPTIONS

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

In [ ]:
model = dq_resnet20(bit_options=BIT_OPTIONS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

LAMBDA = 0.01
EPOCHS = 200
TEMP_START = 1.0
TEMP_END = 0.1
fp32_max = max_bitops(model.layer_flops)

In [ ]:
train_losses, train_accs, test_accs, avg_bitops_log = [], [], [], []

for epoch in range(EPOCHS):
    model.train()
    temp = TEMP_START - (TEMP_START - TEMP_END) * epoch / max(EPOCHS - 1, 1)
    running_loss, correct, total, epoch_bitops = 0, 0, 0, 0

    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs, bits_list = model(inputs, temperature=temp)
        ce_loss = criterion(outputs, targets)
        bitops = compute_bitops(bits_list, model.layer_flops)
        bitops_penalty = bitops / fp32_max
        loss = ce_loss + LAMBDA * bitops_penalty
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        epoch_bitops += bitops.item()

    scheduler.step()
    train_acc = 100. * correct / total
    train_losses.append(running_loss / len(trainloader))
    train_accs.append(train_acc)

    model.eval()
    t_correct, t_total = 0, 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs, _ = model(inputs, temperature=temp)
            _, predicted = outputs.max(1)
            t_total += targets.size(0)
            t_correct += predicted.eq(targets).sum().item()
    test_acc = 100. * t_correct / t_total
    test_accs.append(test_acc)
    avg_bo = epoch_bitops / len(trainloader) / fp32_max
    avg_bitops_log.append(avg_bo)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_losses[-1]:.3f} | Train: {train_acc:.2f}% | Test: {test_acc:.2f}% | AvgBitOPs: {avg_bo:.4f} | Temp: {temp:.3f}')

In [ ]:
torch.save(model.state_dict(), 'dq_resnet20.pth')
print('Model saved.')

## Evaluation


In [ ]:
model.eval()
all_bits = []
correct, total = 0, 0

with torch.no_grad():
    for inputs, targets in testloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs, bits_list = model(inputs)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        all_bits.append([b.item() for b in bits_list])

test_acc = 100. * correct / total
mean_bits = np.mean(all_bits, axis=0)
overall_mean = np.mean(all_bits)

print(f'Test Accuracy: {test_acc:.2f}%')
print(f'Overall Mean Bits: {overall_mean:.2f}')
print(f'\nPer-layer mean bits:')
layer_names = ['conv1'] + [f'L{g+1}.B{b}' for g, n in enumerate([3,3,3]) for b in range(n)]
for name, mb in zip(layer_names, mean_bits):
    print(f'  {name}: {mb:.2f}')

In [ ]:
avg_bitops_test = []
with torch.no_grad():
    for inputs, _ in testloader:
        inputs = inputs.to(device)
        _, bits_list = model(inputs)
        avg_bitops_test.append(compute_bitops(bits_list, model.layer_flops).item() / fp32_max)

print(f'Average BitOPs (relative to FP32): {np.mean(avg_bitops_test):.4f}')

In [ ]:
times = []
model.eval()
with torch.no_grad():
    for inputs, _ in testloader:
        inputs = inputs.to(device)
        _ = model(inputs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    for _ in range(30):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        for inputs, _ in testloader:
            inputs = inputs.to(device)
            _ = model(inputs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.time() - start)

print(f'Avg Inference Time (30 runs): {np.mean(times):.4f} ± {np.std(times):.4f}s')

## Training Curves & Bit Distribution


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(train_accs, label='Train')
axes[0, 0].plot(test_accs, label='Test')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy (%)')
axes[0, 0].set_title('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(train_losses)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Training Loss')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(avg_bitops_log)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('BitOPs (relative to FP32)')
axes[1, 0].set_title('Average BitOPs During Training')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].bar(layer_names, mean_bits, color='#2196F3')
axes[1, 1].set_xlabel('Layer')
axes[1, 1].set_ylabel('Mean Bits')
axes[1, 1].set_title('Per-Layer Mean Bit-Width (Test Set)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('dqnet_results.png', dpi=150, bbox_inches='tight')
plt.show()